# Actividad 6 · Llevando los LLM a producción

**Caso de estudio:** asistente documental universitario con RAG sobre PDF  
**Estudiantes:** Carlos Arnulfo Cogua Laverde, Michael Lan Alvarez y Marcela Gomez Garzon  
**Curso:** LLM · Maestría en Inteligencia Artificial · Universidad de La Salle  
**Fecha:** 16 de noviembre de 2026

Este notebook documenta una aplicación Streamlit con LangChain, OpenRouter, cadena LCEL, RAG sobre documentos PDF, una herramienta documental, filtro local de alcance y memoria conversacional. Cada respuesta debe provenir del corpus y citar el archivo y la página recuperada.

## 1. Objetivo y trazabilidad de la rúbrica

| Criterio | Evidencia |
|---|---|
| Código estructurado y documentado | Módulos en src, README, pruebas y este notebook |
| Funcionalidad y rendimiento | Streamlit, cadena LCEL, agente, RAG, tools, memoria, pytest y benchmark 1/2/5 |
| Ética y sesgos | Sección 8 y docs/ETICA.md |
| Resultados y conclusiones | Secciones 6, 7 y 9 |

**Estado del despliegue público:** pendiente de autorización. La aplicación local no se presenta como producción pública hasta verificar una URL desde otro navegador.

## 2. Arquitectura

Navegador → filtro local de alcance → agente LangChain → consultar_documentos → OpenRouter. Los PDF de data/pdfs se extraen por página, se dividen en fragmentos y se indexan con TF-IDF. Las solicitudes de redacción, código, medios y tareas generales se rechazan sin invocar el modelo. Streamlit asigna un thread_id distinto a cada sesión; InMemorySaver conserva el contexto sólo mientras vive el proceso.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from langchain_core.language_models.fake_chat_models import FakeListChatModel
from src.chains import build_definition_chain
from src.pdf_knowledge import PDF_DIRECTORY, pdf_knowledge_store
from src.tools import consultar_documentos

print('Proyecto:', PROJECT_ROOT)
print('Clave configurada:', bool(os.getenv('OPENROUTER_API_KEY')))


## 3. Cadena LCEL

La cadena enlaza un ChatPromptTemplate, un modelo y StrOutputParser. La prueba siguiente usa un modelo falso determinista para demostrar el flujo sin consumir API; la aplicación usa el modelo elegido de OpenRouter.

In [ ]:
modelo_determinista = FakeListChatModel(
    responses=['Un agente decide qué herramienta utilizar para resolver una solicitud.']
)
cadena = build_definition_chain(modelo_determinista)
print(cadena.invoke({'concepto': 'un agente con herramientas'}))


## 4. RAG y herramientas

Los PDF se cargan desde data/pdfs, se extraen por página y se dividen en fragmentos solapados. TF-IDF permite una recuperación local, reproducible y sin embeddings pagados. Su principal límite es léxico: no comprende bien sinónimos ni garantiza pertinencia.

In [ ]:
pdf_status = pdf_knowledge_store.status()
print('Carpeta de PDF:', PDF_DIRECTORY)
print('Archivos:', pdf_status.files or 'ninguno')
print('Páginas con texto:', pdf_status.pages_with_text)
if pdf_status.files:
    print('\nRecuperación documental:')
    print(consultar_documentos.invoke({'pregunta': '¿Cuál es el tema principal?'}))
else:
    print('Copia los documentos en data/pdfs y vuelve a ejecutar esta celda.')


## 5. Agente y memoria

La ejecución en vivo es opcional y sólo ocurre si OPENROUTER_API_KEY existe en el entorno. La clave nunca se solicita como argumento, se imprime ni se guarda en el notebook.

In [ ]:
if os.getenv('OPENROUTER_API_KEY'):
    from src.agent import build_agent, run_agent
    model_id = os.getenv('OPENROUTER_MODEL', 'openrouter/free')
    agent = build_agent(os.environ['OPENROUTER_API_KEY'], model_id)
    thread_id = 'notebook-demostracion'
    primera = run_agent(agent, '¿Cuánto dura el préstamo de computadores?', thread_id, model_id)
    segunda = run_agent(agent, '¿Y se puede renovar?', thread_id, model_id)
    print(primera.text)
    print(segunda.text)
    print('Herramientas del segundo turno:', segunda.tools)
else:
    print('NO EJECUTADO: falta OPENROUTER_API_KEY. Las pruebas locales continúan disponibles.')


## 6. Pruebas reproducibles

Las pruebas no consumen API. Validan recuperación, herramientas, catálogo de modelos, cadena LCEL, observabilidad del turno actual y paso del thread_id aislado.

In [ ]:
import pytest

exit_code = pytest.main([str(PROJECT_ROOT / 'tests'), '-q'])
assert exit_code == 0


## 7. Rendimiento y cargas 1, 2 y 5

El benchmark real se ejecuta con scripts/benchmark.py y genera CSV de muestras y JSON con tasa de éxito, fallos, latencia media, p50 y p95. Mientras esos archivos no existan, el estado correcto es NO MEDIDO; no se simulan cifras.

In [ ]:
summary_path = PROJECT_ROOT / 'results' / 'benchmark_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
else:
    print('NO MEDIDO: ejecuta scripts/benchmark.py con una clave válida y autorización.')


## 8. Ética, privacidad y sesgos

- **Alucinación:** el modelo puede inventar hechos. Se mitiga con RAG, citas de archivo y página, temperatura cero, instrucciones para reconocer ausencias y pruebas; la verificación humana sigue siendo necesaria.
- **Cobertura y sesgo:** el asistente sólo conoce los PDF cargados. TF-IDF favorece coincidencias literales y puede perjudicar formulaciones con sinónimos. No se deben extrapolar resultados.
- **Privacidad:** los fragmentos recuperados se envían al proveedor del modelo. No deben cargarse documentos confidenciales o con datos sensibles sin autorización y controles adecuados.
- **Disponibilidad y coste:** los modelos gratuitos cambian y tienen cuotas. Los errores 401, 429 y timeout deben comunicarse sin prometer servicio continuo.
- **Límite de uso:** el prototipo no debe decidir admisiones, beneficios, sanciones ni otros asuntos de alto impacto.

## 9. Interpretación y conclusiones

La evidencia local demuestra que los componentes deterministas funcionan y que la aplicación arranca por HTTP. La corrección de herramientas y recuperación no garantiza por sí sola la calidad de todas las respuestas del LLM. La memoria es adecuada para una demostración, no para persistencia durable.

La conclusión sobre rendimiento queda condicionada a una medición real contra un modelo y una fecha concretos. Antes de afirmar producción completa faltan: ejecutar y analizar el benchmark autorizado, desplegar en Streamlit Community Cloud, verificar la URL en incógnito y registrar esas evidencias en este notebook.

## Fuentes

- LangChain, documentación de create_agent y memoria de corto plazo.
- OpenRouter, catálogo y documentación de modelos.
- Streamlit, documentación de Session State y Community Cloud.
- Guía de la Actividad 6 suministrada en el repositorio local.